In [1]:
#imports
import sys; sys.path.append("..")
import numpy as np
from scipy.io import loadmat

In [2]:
with open(r"C:\spike-denoising-synthetic\data\real\_Information.txt") as f:
    print(f.read())

2025/10/06
This folder contains time series data recorded from the saphenous nerve of a rat, including the filtered data signal and the 'ground truth' time points for two single nerve units.
- time_series_2.bin : the time information of the data recordings
- 100_RhythmData_...bin : the data recordings from several relevant channels. This data has been common average referenced (median), and filtered (300-6000Hz), and stimulation artifacts have been blanked (i.e., set to the mean).
- grouped_unit_1.mat : describes the time points of a 'good' unit which has large spikes on multiple data channels
- grouped_unit_2.mat : describes the time points of a 'bad' unit which has spikes on only one data channel

The 'grouped_unit_' files contain a struct, most of which is not useful. The relevant information is under the 'CH**' field (i.e., whichever field the spike was recorded on, there may be multiple of these fields). Under the 'CH**' field is the 'aligned_zc_locs' which will describe when (i.e

In [3]:
with open(r"C:\spike-denoising-synthetic\data\real\example_script.m") as f:
    print(f.read())

% Script to load example data

% Change the string below to navigate to the correct folder/directory
folder_path = "C:\Users\...\2025-02-13 - Data for sharing";

% Load the timestamps
fid = fopen(folder_path + "\time_series_2.bin");
time_stamps = fread(fid,inf,'double');
fclose(fid);

% Load the grouped unit struct
grouped_unit = load(folder_path + "\grouped_unit_1.mat");
field_names = fieldnames(grouped_unit);

% For each of the channels ('CH**'), graph a period of time to show spikes
channels_of_interest = field_names(contains(field_names,"CH"));
time_block_of_interest = [3000 3360]; % the time period we will graph
selected_timestamps = time_stamps(time_stamps>time_block_of_interest(1) & time_stamps<time_block_of_interest(2));

figure;
hold on;
for i=1:length(channels_of_interest)
    % load the data recording
    if ~isfile(folder_path + "\100_RhythmData_"+channels_of_interest{i}+"_2_filtered_blanked.bin")
        continue
    end
    fid = fopen(folder_path + "\100_RhythmData_"+cha

In [4]:
path = r"C:\spike-denoising-synthetic\data\real"

# read the first 2 million samples of CH 19
n_read = 2_000_000
ch19 = np.fromfile(path + r"\100_RhythmData_CH19_2_filtered_blanked.bin",
                   dtype='<f8', count=n_read)

# rad the matching timestamps 
tstamps = np.fromfile(path +  r"\time_series_2.bin",
                      dtype='<f8', count=n_read)

print("ch19 samples:", ch19.shape)
print("ch19 range:", ch19.min(), "to", ch19.max())
print("timestamps range:", tstamps.min(), "to", tstamps.max())
print("first 5 timestamps:", tstamps[:5])

ch19 samples: (2000000,)
ch19 range: -23.76011994785937 to 23.91417943875254
timestamps range: 1.03 to 67.69663333333334
first 5 timestamps: [1.03       1.03003333 1.03006667 1.0301     1.03013333]


In [5]:
print(tstamps[1] - tstamps[0])

3.333333333332966e-05


In [6]:
print("any nan or inf?", np.isnan(ch19).any(), np.isinf(ch19).any())
print("mean:", ch19.mean())
print("std:", ch19.std())
print("max abs value:", np.abs(ch19).max())

any nan or inf? False False
mean: -0.00038347341719193004
std: 4.170475189093898
max abs value: 23.91417943875254


In [8]:
import h5py
print(h5py.__version__)

3.15.1


In [10]:
gt = h5py.File(r"C:\spike-denoising-synthetic\data\real\grouped_unit_1.mat", 'r')
print("keys:", list(gt.keys()))

keys: ['#refs#', '#subsystem#', 'CH10', 'CH14', 'CH19', 'CH25', 'CH3', 'CH30', 'CH9', 'filename', 'group_number', 'primary_channel']


In [11]:
print("fields inside CH19:", list(gt['CH19'].keys()))

fields inside CH19: ['aligned_zc_lats', 'aligned_zc_locs', 'channel', 'config', 'experiment', 'filename', 'highlighted_data_TTLs', 'highlighted_data_lats', 'highlighted_data_logs', 'latencies', 'mech_after', 'mech_before', 'nth_TTL', 'plot_handles', 'probe_axis', 'probe_data', 'spike_plot_data', 'spikes_axis', 'status', 'traces_axis', 'true_locs', 'zc_alignment', 'zc_locs']


In [13]:
zc = gt['CH19']['aligned_zc_locs'][:]
print("shape:", zc.shape)
print("how many spikes:", zc.size)
print("first few:", zc.flatten()[:5])
print("range:", zc.min(), "to", zc.max())

shape: (1, 1623)
how many spikes: 1623
first few: [1754.45036667 1758.45       1762.4505     1766.45126667 1770.45166667]
range: 1754.4503666666667 to 8265.239733333334


In [14]:
zc_times = zc.flatten()   # the 4 spike times, in seconds
print("spike times:", zc_times)

# how far into the file does our loaded chunk reach?
print("chunk covers 0 to", n_read / 30000, "seconds")

spike times: [1754.45036667 1758.45       1762.4505     ... 8257.23826667 8261.23906667
 8265.23973333]
chunk covers 0 to 66.66666666666667 seconds


In [16]:
zc_times = zc.flatten()
print("spike times (s):", zc_times)

spike times (s): [1754.45036667 1758.45       1762.4505     ... 8257.23826667 8261.23906667
 8265.23973333]


In [17]:
fs_real = 30000
t_spike = zc_times[0]                      # first spike time, seconds
idx = int(t_spike * fs_real)               # its sample position in the file

# load a window around it: 15000 samples (~0.5 s) starting a bit before the spike
start = idx - 7500
n_win = 15000
ch19_seg = np.fromfile(
    path + r"\100_RhythmData_CH19_2_filtered_blanked.bin",
    dtype='<f8', count=n_win, offset=start * 8)   # *8 because float64 = 8 bytes

# the spike sits at sample 7500 within this segment
local_idx = idx - start
window = ch19_seg[local_idx-45 : local_idx+45]
print("trough at spike:", window.min())
print("segment std:", ch19_seg.std())
print("trough / std:", window.min() / ch19_seg.std())

trough at spike: -12.073138290313675
segment std: 5.6119061219814546
trough / std: -2.1513435948302866


In [18]:
mad = np.median(np.abs(ch19_seg - np.median(ch19_seg))) / 0.6745
print("trough / MAD:", window.min() / mad)

trough / MAD: -2.296081559392508
